# Part 0 - Pre-quantize Llama-2-7B-chat to INT4 (bitsandbytes)

Loads `meta-llama/Llama-2-7b-chat-hf` once, applies bitsandbytes 4-bit quantization,
and saves the resulting checkpoint to `./quantized_int4/`. All later notebooks
(02 / 04b / 05 / 06 / 07 / 08 / 09) load **from this local path** instead of from HF, via
the path-rewriting layer added to `jbb_common.patch_jbb()`.

Run this **first**, in its own kernel. Idempotent: re-running with the checkpoint already on
disk is a no-op.

**Output**: `./quantized_int4/` (model + tokenizer files).


In [ ]:
import os, sys
from pathlib import Path

# Self-locate jbb_common.py so kernel CWD doesn't matter.
_cwd = Path.cwd()
if (_cwd / "jbb_common.py").exists():
    pass
elif (_cwd / "tml" / "jbb_common.py").exists():
    os.chdir(_cwd / "tml")
else:
    raise FileNotFoundError(f"jbb_common.py not found from {_cwd}; run from tml/ or repo root.")
sys.path.insert(0, str(Path.cwd()))

from jbb_common import hf_login
hf_login()


In [ ]:
# Pre-quantize Llama-2-7B-chat-hf to INT4 (bitsandbytes) and save locally.
# All downstream notebooks redirect FP16 target loads to this checkpoint via jbb_common.patch_jbb().
# Run this notebook ONCE on the cluster before 02 / 04b / 05 / 06 / 07 / 08 / 09.

import gc
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SRC_MODEL = "meta-llama/Llama-2-7b-chat-hf"
OUT_DIR   = Path("./quantized_int4")

if (OUT_DIR / "config.json").exists():
    print(f"Already quantized at {OUT_DIR.resolve()} - skipping.")
else:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    print(f"Quantizing {SRC_MODEL} -> INT4 (bitsandbytes)...")
    model = AutoModelForCausalLM.from_pretrained(
        SRC_MODEL,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    tok = AutoTokenizer.from_pretrained(SRC_MODEL)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(OUT_DIR)
    tok.save_pretrained(OUT_DIR)
    print(f"Saved INT4 checkpoint to {OUT_DIR.resolve()}")
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()
